In [ ]:
from pathlib import Path
import requests
from tqdm import tqdm
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env into environment

APIKEY = os.getenv("DATAFORDELER_APIKEY")
if not APIKEY:
    raise RuntimeError("Missing DATAFORDELER_APIKEY in .env")

BASE = "https://api.datafordeler.dk"
REGISTER = "GeoDKO"  # GeoDanmark Ortofoto register in raster download API docs
OUT = Path("geodko_tiles")
OUT.mkdir(exist_ok=True)

def get_available_tiles():
    url = f"{BASE}/FileDownloads/GetAvailableRasterFileDownloads"
    params = {"Register": REGISTER, "apikey": APIKEY}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    return r.json()

def download_one(filename: str):
    url = f"{BASE}/FileDownloads/GetRasterFile"
    params = {"Register": REGISTER, "Filename": filename, "apikey": APIKEY}
    out_path = OUT / filename

    if out_path.exists():
        return

    out_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.get(url, params=params, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

def extract_filenames(tile_list):
    """
    Datafordeler's JSON field names can vary. This tries a few common keys.
    Print tile_list[0] once if this fails and adjust here.
    """
    if not tile_list:
        return []

    candidate_keys = ["Filename", "filename", "FileName", "name"]
    for k in candidate_keys:
        if isinstance(tile_list[0], dict) and k in tile_list[0]:
            return [t[k] for t in tile_list if k in t]

    raise KeyError(f"Couldn't find a filename key. First item was: {tile_list[0]}")

def main():
    tiles = get_available_tiles()
    filenames = extract_filenames(tiles)

    print("Tiles found:", len(filenames))
    print("Example filename:", filenames[0] if filenames else None)

    for fn in tqdm(filenames, desc="Downloading tiles"):
        download_one(fn)

    print("Done. Saved to:", OUT.resolve())

if __name__ == "__main__":
    main()


## Take a peak

In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
import zipfile
import requests
import time

load_dotenv()
APIKEY = os.getenv("DATAFORDELER_APIKEY")
if not APIKEY:
    raise RuntimeError("Missing DATAFORDELER_APIKEY in .env")

BASE = "https://api.datafordeler.dk/FileDownloads"

OUTDIR = Path("geodko10cm_one_tile")
OUTDIR.mkdir(parents=True, exist_ok=True)

def list_one_tile():
    url = f"{BASE}/GetAvailableRasterFileDownloads"
    params = {
        "apiKey": APIKEY,
        "Register": "GeoDKO",
        "DataSetName": "GeoDKO10cm",
        "Version": 1,
        "FileFormat": "tif",
        "PageNumber": 1,
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    items = data.get("availableFileDownloads") or data.get("AvailableFileDownloads")
    if not items:
        raise RuntimeError(f"No tiles returned. Response keys: {list(data.keys())}")

    return items[0]

def sniff_file_type(path: Path) -> str:
    """
    Detect common formats by magic bytes:
      ZIP: 50 4B 03 04  => b'PK\\x03\\x04'
      TIFF: 'II*\\x00' (little endian) or 'MM\\x00*' (big endian)
    """
    head = path.read_bytes()[:4]
    if head.startswith(b"PK\x03\x04"):
        return "zip"
    if head in (b"II*\x00", b"MM\x00*"):
        return "tif"
    return "unknown"

def download_tile(tile_meta):
    url = f"{BASE}/GetRasterFile"

    tile_id = tile_meta.get("id") or tile_meta.get("ID")
    file_name = tile_meta.get("fileName") or tile_meta.get("FileName")
    if not tile_id and not file_name:
        raise RuntimeError(f"Missing id/fileName in tile_meta: {tile_meta}")

    # Use the official parameter name from docs: apiKey
    params = {"apiKey": APIKEY}
    if tile_id:
        params["Id"] = tile_id
        base_name = str(tile_id)
    else:
        params["FileName"] = file_name
        base_name = str(file_name).replace("/", "_")

    tmp_path = OUTDIR / f"{base_name}.download"

    start = time.time()
    bytes_dl = 0

    with requests.get(url, params=params, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                bytes_dl += len(chunk)

                elapsed = time.time() - start
                if elapsed > 0:
                    mb_s = (bytes_dl / (1024**2)) / elapsed
                    print(f"\rDownloaded {bytes_dl/1024**2:.1f} MiB @ {mb_s:.1f} MiB/s", end="")
    print()

    ftype = sniff_file_type(tmp_path)
    content_type = None
    try:
        # optional: look at server content-type for debugging
        # (not always reliable, but nice to print)
        content_type = requests.get(url, params=params, timeout=10).headers.get("Content-Type")
    except Exception:
        pass

    print("Detected downloaded file type:", ftype, "| Content-Type:", content_type)

    if ftype == "zip":
        zip_path = OUTDIR / f"{base_name}.zip"
        tmp_path.rename(zip_path)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(OUTDIR)

        return {"kind": "zip", "path": zip_path}

    if ftype == "tif":
        tif_path = OUTDIR / (file_name if file_name else f"{base_name}.tif")
        # ensure suffix
        if not str(tif_path).lower().endswith(".tif"):
            tif_path = tif_path.with_suffix(".tif")

        tmp_path.rename(tif_path)
        return {"kind": "tif", "path": tif_path}

    # Unknown: dump first bytes as text so you can see if it's an error page
    preview = tmp_path.read_bytes()[:600]
    try:
        preview_text = preview.decode("utf-8", errors="replace")
    except Exception:
        preview_text = repr(preview)

    raise RuntimeError(
        "Download was neither ZIP nor TIFF. This often means you saved an error page.\n"
        f"First bytes preview:\n{preview_text}"
    )

def main():
    tile = list_one_tile()
    print("Chosen tile metadata:")
    print(tile)

    result = download_tile(tile)
    print("\nDownload result:", result["kind"], result["path"])

    tifs = list(OUTDIR.rglob("*.tif"))
    print("GeoTIFF files found:", [p.name for p in tifs])

if __name__ == "__main__":
    main()


Chosen tile metadata:
{'fileFormat': 'tif', 'id': '00004ce5-afe1-4038-bedd-38b00e3c5313', 'dataSetName': 'GeoDKO10cm', 'register': 'geodko', 'version': '1', 'dataDeliveryNumber': 4, 'fileName': '2025_1km_6148_579.tif', 'generationTime': '2026-01-12T16:14:20.065608Z', 'boundingBox': {'minX': 579000, 'maxX': 580000, 'minY': 6148000, 'maxY': 6149000}, 'fileSizeInBytes': 106847662}
Downloaded 101.9 MiB @ 2.4 MiB/s
Detected downloaded file type: tif | Content-Type: application/zip

Download result: tif geodko10cm_one_tile/2025_1km_6148_579.tif
GeoTIFF files found: ['2025_1km_6148_579.tif']


## Hvor meget fylder det hele? 10cm

In [8]:
from dotenv import load_dotenv
import os
import requests
import time

load_dotenv()
APIKEY = os.getenv("DATAFORDELER_APIKEY")
if not APIKEY:
    raise RuntimeError("Missing DATAFORDELER_APIKEY in .env")

URL = "https://api.datafordeler.dk/FileDownloads/GetAvailableRasterFileDownloads"

def human_bytes(n: int) -> str:
    gib = n / (1024**3)
    if gib < 1024:
        return f"{gib:.2f} GiB"
    return f"{n / (1024**4):.2f} TiB"

def fetch_page(page_number: int):
    params = {
        "apiKey": APIKEY,
        "Register": "GeoDKO",
        "DataSetName": "GeoDKO10cm",
        "Version": 1,
        "FileFormat": "tif",
        "PageNumber": page_number,
    }
    r = requests.get(URL, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()
    items = data.get("availableFileDownloads") or data.get("AvailableFileDownloads") or []
    meta = data.get("paginationMetadata") or data.get("PaginationMetadata") or {}
    return items, meta

def main():
    t0 = time.time()

    # Page 1 to discover total pages
    items, meta = fetch_page(1)
    total_pages = meta.get("totalPages") or meta.get("TotalPages")
    total_count = meta.get("totalCount") or meta.get("TotalCount")  # optional
    page_size = meta.get("pageSize") or meta.get("PageSize")        # optional

    if not total_pages:
        raise RuntimeError(f"Couldn't find totalPages in paginationMetadata. Keys: {list(meta.keys())}")

    total_bytes = 0
    total_files = 0

    def consume(page_items):
        nonlocal total_bytes, total_files
        for it in page_items:
            total_bytes += int(it.get("fileSizeInBytes", 0) or 0)
        total_files += len(page_items)

    consume(items)

    # Print initial progress
    print(f"Page 1/{total_pages} | files={total_files}"
          + (f"/{total_count}" if total_count else "")
          + f" | summed={human_bytes(total_bytes)}")

    # Loop remaining pages
    for page in range(2, int(total_pages) + 1):
        page_items, _ = fetch_page(page)
        consume(page_items)

        # progress line
        elapsed = time.time() - t0
        pages_done = page
        pct = 100.0 * pages_done / int(total_pages)
        rate_pages = pages_done / elapsed if elapsed > 0 else 0.0

        # estimate remaining time
        remaining_pages = int(total_pages) - pages_done
        eta = remaining_pages / rate_pages if rate_pages > 0 else float("inf")

        print(
            f"Page {page}/{total_pages} ({pct:5.1f}%) | "
            f"files={total_files}" + (f"/{total_count}" if total_count else "") +
            f" | summed={human_bytes(total_bytes)} | "
            f"{rate_pages:.2f} pages/s | ETA {eta/60:.1f} min"
        )

    avg = total_bytes // max(total_files, 1)
    print("\nDONE")
    print("Files:", total_files)
    print("Total size:", human_bytes(total_bytes))
    print("Average per file:", human_bytes(avg))

if __name__ == "__main__":
    main()


Page 1/193 | files=100/19203 | summed=10.47 GiB
Page 2/193 (  1.0%) | files=200/19203 | summed=21.28 GiB | 2.01 pages/s | ETA 1.6 min
Page 3/193 (  1.6%) | files=300/19203 | summed=31.71 GiB | 1.37 pages/s | ETA 2.3 min
Page 4/193 (  2.1%) | files=400/19203 | summed=42.11 GiB | 1.47 pages/s | ETA 2.1 min
Page 5/193 (  2.6%) | files=500/19203 | summed=52.66 GiB | 1.48 pages/s | ETA 2.1 min
Page 6/193 (  3.1%) | files=600/19203 | summed=63.79 GiB | 1.38 pages/s | ETA 2.3 min
Page 7/193 (  3.6%) | files=700/19203 | summed=75.05 GiB | 1.14 pages/s | ETA 2.7 min
Page 8/193 (  4.1%) | files=800/19203 | summed=85.72 GiB | 1.19 pages/s | ETA 2.6 min
Page 9/193 (  4.7%) | files=900/19203 | summed=96.67 GiB | 0.99 pages/s | ETA 3.1 min
Page 10/193 (  5.2%) | files=1000/19203 | summed=107.69 GiB | 1.02 pages/s | ETA 3.0 min
Page 11/193 (  5.7%) | files=1100/19203 | summed=118.11 GiB | 1.04 pages/s | ETA 2.9 min
Page 12/193 (  6.2%) | files=1200/19203 | summed=128.61 GiB | 1.03 pages/s | ETA 2.9 m

## Hvor meget fylder det hele? 12,5cm

In [10]:
from dotenv import load_dotenv
import os
import requests
import time

load_dotenv()
APIKEY = os.getenv("DATAFORDELER_APIKEY")
if not APIKEY:
    raise RuntimeError("Missing DATAFORDELER_APIKEY in .env")

URL = "https://api.datafordeler.dk/FileDownloads/GetAvailableRasterFileDownloads"

def human_bytes(n: int) -> str:
    gib = n / (1024**3)
    if gib < 1024:
        return f"{gib:.2f} GiB"
    return f"{n / (1024**4):.2f} TiB"

def fetch_page(page_number: int):
    params = {
        "apiKey": APIKEY,
        "Register": "GeoDKO",
        "DataSetName": "GeoDKO12,5cm",
        "Version": 1,
        "FileFormat": "tif",
        "PageNumber": page_number,
    }
    r = requests.get(URL, params=params, timeout=1200)
    r.raise_for_status()
    data = r.json()
    items = data.get("availableFileDownloads") or data.get("AvailableFileDownloads") or []
    meta = data.get("paginationMetadata") or data.get("PaginationMetadata") or {}
    return items, meta

def main():
    t0 = time.time()

    # Page 1 to discover total pages
    items, meta = fetch_page(1)
    total_pages = meta.get("totalPages") or meta.get("TotalPages")
    total_count = meta.get("totalCount") or meta.get("TotalCount")  # optional
    page_size = meta.get("pageSize") or meta.get("PageSize")        # optional

    if not total_pages:
        raise RuntimeError(f"Couldn't find totalPages in paginationMetadata. Keys: {list(meta.keys())}")

    total_bytes = 0
    total_files = 0

    def consume(page_items):
        nonlocal total_bytes, total_files
        for it in page_items:
            total_bytes += int(it.get("fileSizeInBytes", 0) or 0)
        total_files += len(page_items)

    consume(items)

    # Print initial progress
    print(f"Page 1/{total_pages} | files={total_files}"
          + (f"/{total_count}" if total_count else "")
          + f" | summed={human_bytes(total_bytes)}")

    # Loop remaining pages
    for page in range(2, int(total_pages) + 1):
        page_items, _ = fetch_page(page)
        consume(page_items)

        # progress line
        elapsed = time.time() - t0
        pages_done = page
        pct = 100.0 * pages_done / int(total_pages)
        rate_pages = pages_done / elapsed if elapsed > 0 else 0.0

        # estimate remaining time
        remaining_pages = int(total_pages) - pages_done
        eta = remaining_pages / rate_pages if rate_pages > 0 else float("inf")

        print(
            f"Page {page}/{total_pages} ({pct:5.1f}%) | "
            f"files={total_files}" + (f"/{total_count}" if total_count else "") +
            f" | summed={human_bytes(total_bytes)} | "
            f"{rate_pages:.2f} pages/s | ETA {eta/60:.1f} min"
        )

    avg = total_bytes // max(total_files, 1)
    print("\nDONE")
    print("Files:", total_files)
    print("Total size:", human_bytes(total_bytes))
    print("Average per file:", human_bytes(avg))

if __name__ == "__main__":
    main()


Page 1/1029 | files=100/102871 | summed=6.55 GiB
Page 2/1029 (  0.2%) | files=200/102871 | summed=13.15 GiB | 0.20 pages/s | ETA 85.2 min
Page 3/1029 (  0.3%) | files=300/102871 | summed=19.64 GiB | 0.28 pages/s | ETA 60.5 min
Page 4/1029 (  0.4%) | files=400/102871 | summed=26.03 GiB | 0.36 pages/s | ETA 46.9 min
Page 5/1029 (  0.5%) | files=500/102871 | summed=32.61 GiB | 0.43 pages/s | ETA 39.8 min
Page 6/1029 (  0.6%) | files=600/102871 | summed=39.08 GiB | 0.49 pages/s | ETA 34.5 min
Page 7/1029 (  0.7%) | files=700/102871 | summed=45.52 GiB | 0.56 pages/s | ETA 30.4 min
Page 8/1029 (  0.8%) | files=800/102871 | summed=51.96 GiB | 0.62 pages/s | ETA 27.6 min
Page 9/1029 (  0.9%) | files=900/102871 | summed=58.44 GiB | 0.66 pages/s | ETA 25.8 min
Page 10/1029 (  1.0%) | files=1000/102871 | summed=64.95 GiB | 0.70 pages/s | ETA 24.1 min
Page 11/1029 (  1.1%) | files=1100/102871 | summed=71.38 GiB | 0.75 pages/s | ETA 22.7 min
Page 12/1029 (  1.2%) | files=1200/102871 | summed=78.05 

ReadTimeout: HTTPSConnectionPool(host='api.datafordeler.dk', port=443): Read timed out. (read timeout=1200)